# Parte 5: Verificación App vs Notebook

**Objetivo:** demostrar que las mismas filas del conjunto de test dan el mismo resultado en la app y aquí.

1. Se carga el mismo `pipeline_ganador.joblib` que usa la app.
2. Se predicen las filas de test directamente, sin pasar por Streamlit.
3. Se comparan contra `resultados_app_*.csv`, descargados desde la app tras subir esas filas.

In [1]:
import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import joblib
import numpy as np
import pandas as pd
import sklearn

from src.data_prep import get_train_test
from src.preprocessor import ALL_FEATURE_COLS
from src.scoring import metricas, semaforo

EVIDENCIA = ROOT_DIR / 'reports' / 'evidencia_viii4'

X_train, X_test, y_train, y_test = get_train_test()
X_test = X_test[ALL_FEATURE_COLS]
pipeline = joblib.load(ROOT_DIR / 'models' / 'pipeline_ganador.joblib')
umbral = json.loads((ROOT_DIR / 'models' / 'metricas_test.json').read_text(encoding='utf-8'))['umbral_operativo']

print(f'scikit-learn {sklearn.__version__} | X_test: {X_test.shape} | umbral operativo: {umbral}')

scikit-learn 1.9.1 | X_test: (885, 36) | umbral operativo: 0.4


## 1. Modelo que usa la app

In [2]:
clf = pipeline.named_steps['classifier']
print('Modelo:', type(clf).__name__)
print({k: clf.get_params()[k] for k in ['n_estimators', 'learning_rate', 'max_depth', 'subsample', 'random_state']})
print('Columnas tras el preprocesamiento:', len(pipeline.named_steps['preprocessor'].get_feature_names_out()))
pipeline

Modelo: GradientBoostingClassifier
{'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 0.8, 'random_state': 42}
Columnas tras el preprocesamiento: 235


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](36,)","['Marital Status','Application mode','Application order',..., 'Unemployment rate','Inflation rate','GDP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,36
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nominal', ...), ('binary', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that gen

## 2. Rendimiento sobre el conjunto de test

In [3]:
proba = pd.Series(pipeline.predict_proba(X_test)[:, 1], index=X_test.index)

for valor in (0.50, umbral):
    m = metricas(y_test, proba.values, valor)
    print(f"Umbral {valor:.2f} | Accuracy {m['accuracy']:.4f} | Precision {m['precision']:.4f} | "
          f"Recall {m['recall']:.4f} | F1 {m['f1']:.4f} | ROC-AUC {m['roc_auc']:.4f} | "
          f"TN, FP, FN, TP = {m['cm'].ravel().tolist()}")

Umbral 0.50 | Accuracy 0.8836 | Precision 0.8606 | Recall 0.7606 | F1 0.8075 | ROC-AUC 0.9343 | TN, FP, FN, TP = [566, 35, 68, 216]
Umbral 0.40 | Accuracy 0.8881 | Precision 0.8223 | Recall 0.8310 | F1 0.8266 | ROC-AUC 0.9343 | TN, FP, FN, TP = [550, 51, 48, 236]


## 3. Las mismas 10 filas en el notebook y en la app

Filas fijadas sin mirar probabilidades: primeras 5 con `dropout = 1` y primeras 5 con `dropout = 0`, en el orden del test. Están en `filas_paridad_10.csv`, el archivo que se subió a la app.

In [4]:
filas = pd.read_csv(EVIDENCIA / 'filas_paridad_10.csv').set_index('id_estudiante')
app = pd.read_csv(EVIDENCIA / 'resultados_app_10.csv').set_index('id_estudiante')

tabla = pd.DataFrame({
    'Notebook: probabilidad': proba[filas.index].round(4),
    'App: probabilidad': app['Prob. Deserción'].reindex(filas.index),
    'Notebook: semáforo': semaforo(proba[filas.index], umbral),
    'App: semáforo': app['Semáforo'].reindex(filas.index),
    'Resultado real (1 = abandonó)': y_test[filas.index],
}).sort_values('Notebook: probabilidad', ascending=False)
tabla.index.name = 'ID'
tabla['¿Coincide?'] = np.where(
    (tabla['Notebook: probabilidad'] == tabla['App: probabilidad'])
    & (tabla['Notebook: semáforo'] == tabla['App: semáforo']), 'Sí', 'No'
)
tabla

,Notebook: probabilidad,App: probabilidad,Notebook: semáforo,App: semáforo,Resultado real (1 = abandonó),¿Coincide?
ID,,,,,,
477,0.9811,0.9811,Rojo,Rojo,1,Sí
2266,0.8943,0.8943,Rojo,Rojo,1,Sí
971,0.8936,0.8936,Rojo,Rojo,1,Sí
3861,0.8366,0.8366,Rojo,Rojo,1,Sí
1220,0.5023,0.5023,Amarillo,Amarillo,1,Sí
2555,0.2169,0.2169,Verde,Verde,0,Sí
3545,0.0444,0.0444,Verde,Verde,0,Sí
269,0.0421,0.0421,Verde,Verde,0,Sí
2892,0.0386,0.0386,Verde,Verde,0,Sí


## 4. Comparación completa: 885 filas de test

In [5]:
app_885 = pd.read_csv(EVIDENCIA / 'resultados_app_885.csv').set_index('id_estudiante').reindex(X_test.index)

dif = (proba.round(4) - app_885['Prob. Deserción']).abs()
iguales_semaforo = (semaforo(proba, umbral) == app_885['Semáforo'].values).sum()

print(f'Filas comparadas: {len(app_885)} (faltantes en la app: {app_885["Prob. Deserción"].isna().sum()})')
print(f'Diferencia máxima de probabilidad, redondeada a 4 decimales: {dif.max():.4f}')
print(f'Filas con igual probabilidad: {(dif == 0).sum()} de {len(dif)}')
print(f'Filas con igual semáforo: {iguales_semaforo} de {len(dif)}')

assert app_885['Prob. Deserción'].notna().all() and dif.max() == 0 and iguales_semaforo == len(dif)
print('VERIFICACIÓN OK: la app y el notebook dan el mismo resultado.')

Filas comparadas: 885 (faltantes en la app: 0)
Diferencia máxima de probabilidad, redondeada a 4 decimales: 0.0000
Filas con igual probabilidad: 885 de 885
Filas con igual semáforo: 885 de 885
VERIFICACIÓN OK: la app y el notebook dan el mismo resultado.
